# IEEE 118-Bus System - Security-Constrained DC Optimal Power Flow (SC-DCOPF)

Branch-oriented SC-DCOPF following the thesis formulation. Minimize expected generation
cost while keeping the dispatch feasible across a set of single-element ($N-1$) contingencies
$\mathcal{C} = \{0\} \cup \mathcal{C}_T \cup \mathcal{C}_G$.

$$\min_{P,f,\theta} \sum_{g,c} \rho_c\, c_g\, P_{g,c}$$

subject to nodal balance (3.2), generator limits (3.3), thermal limits (3.4), the DC flow
law (3.5)/(3.5b), and the reference-bus constraint (3.6).

All input data is read from `processed/`. Per-unit on a
100 MVA base.

**Build strategy:** we start with the base case only ($\mathcal{C}=\{0\}$) to validate the
mechanics against a plain DCOPF, then switch on $\mathcal{C}_T$ and $\mathcal{C}_G$.

## 1. Imports and processed data

In [19]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB

In [20]:
PROC = "processed"

gens     = pd.read_csv(f"{PROC}/gens.csv")
loads    = pd.read_csv(f"{PROC}/loads.csv")
branches = pd.read_csv(f"{PROC}/branches.csv")
meta     = pd.read_csv(f"{PROC}/system_meta.csv").set_index("key")["value"]

BASE_MVA  = float(meta["base_mva"])
SLACK_BUS = int(meta["slack_bus"])

print(f"base_mva = {BASE_MVA:.0f} MVA | slack bus = {SLACK_BUS}")
print(f"generators = {len(gens)} | buses = {len(loads)} | branches = {len(branches)}")

base_mva = 100 MVA | slack bus = 69
generators = 19 | buses = 118 | branches = 186


## 2. Index sets and branch-oriented structures

We build the sets and lookups the formulation needs. The nodal balance (3.2) is written per
bus using the sets of **outgoing** lines $\Omega_i^+$ (branches with `from_bus == i`) and
**incoming** lines $\Omega_i^-$ (`to_bus == i`), plus the generators attached to each bus
$\mathcal{G}_i$.

In [21]:
# --- sets ---
set_N = loads["bus"].tolist()          # buses      (N)
set_G = gens["gen_id"].tolist()        # generators (G)
set_L = branches["branch_id"].tolist() # branches   (L)

# --- per-element parameters ---
par_pd     = dict(zip(loads["bus"],          loads["pd_pu"]))      # P_{d,i}
par_g_bus  = dict(zip(gens["gen_id"],        gens["bus"]))         # bus of generator g
par_pmin   = dict(zip(gens["gen_id"],        gens["pmin_pu"]))     # P_g^min
par_pmax   = dict(zip(gens["gen_id"],        gens["pmax_pu"]))     # P_g^max
par_cost   = dict(zip(gens["gen_id"],        gens["c1"]))          # c_g  ($/MWh)
par_from   = dict(zip(branches["branch_id"], branches["from_bus"]))
par_to     = dict(zip(branches["branch_id"], branches["to_bus"]))
par_x      = dict(zip(branches["branch_id"], branches["x_pu"]))    # x_{ij}
par_rate   = dict(zip(branches["branch_id"], branches["rate_pu"])) # F_{ij}^max

# --- subset G_i : generators connected to each bus ---
set_gens_at = {i: [] for i in set_N}
for g in set_G:
    set_gens_at[par_g_bus[g]].append(g)

# --- subsets Omega_i^+ (leaving bus i) and Omega_i^- (entering bus i) ---
set_out_lines = {i: [] for i in set_N}   # Omega_i^+  (from_bus == i)
set_in_lines  = {i: [] for i in set_N}   # Omega_i^-  (to_bus == i)
for l in set_L:
    set_out_lines[par_from[l]].append(l)
    set_in_lines[par_to[l]].append(l)

# --- integrity checks ---
assert SLACK_BUS in set_N, "Slack bus not found among buses."
assert sum(len(v) for v in set_gens_at.values()) == len(set_G)
assert sum(len(v) for v in set_out_lines.values()) == len(set_L)

print(f"|N| = {len(set_N)} buses | |G| = {len(set_G)} generators | |L| = {len(set_L)} branches")
print(f"Bus {SLACK_BUS} (slack): gens={set_gens_at[SLACK_BUS]}, "
      f"lines out={len(set_out_lines[SLACK_BUS])}, lines in={len(set_in_lines[SLACK_BUS])}")

|N| = 118 buses | |G| = 19 generators | |L| = 186 branches
Bus 69 (slack): gens=[12], lines out=3, lines in=3


## 3. Operating states $\mathcal{C}$, availability $N1$, and probabilities $\rho$

$\mathcal{C} = \{0\} \cup \mathcal{C}_T \cup \mathcal{C}_G$: the base case plus one scenario per
single-element contingency. $\mathcal{C}_T$ uses **non-radial lines only** (bridges are
excluded, since their outage would island a bus); $\mathcal{C}_G$ uses the real generators.

**Build strategy:** the two switches below start **off**, so $\mathcal{C}=\{0\}$ and the model
reduces to a plain DCOPF for validation. Set them to `True` later to grow to the full
$N-1$ set — nothing else changes.

`N1` is stored implicitly: each scenario records *which* element is out; the helpers
`line_on(l, c)` and `gen_on(g, c)` return the binary availability $N1_{ij,c}$ / $N1_{g,c}$.

In [22]:
INCLUDE_CT = True   # transmission contingencies (non-radial lines)
INCLUDE_CG = True   # generator contingencies

# Eligible contingency elements
CT_lines = branches.loc[~branches["is_bridge"], "branch_id"].tolist()  # non-radial only
CG_gens  = set_G                                                       # all real generators

# Build the scenario list. Each scenario knows which element (if any) is out of service.
scenarios = [{"id": 0, "type": "base", "out_line": None, "out_gen": None,
              "desc": "base case (all in service)"}]

if INCLUDE_CT:
    for l in CT_lines:
        scenarios.append({"id": len(scenarios), "type": "line",
                          "out_line": l, "out_gen": None, "desc": f"line {l} out"})
if INCLUDE_CG:
    for g in CG_gens:
        scenarios.append({"id": len(scenarios), "type": "gen",
                          "out_line": None, "out_gen": g, "desc": f"gen {g} out"})

set_C = [s["id"] for s in scenarios]     # C : operating states
scen  = {s["id"]: s for s in scenarios}  # lookup by scenario id

# N1 availability (0 = out of service, 1 = in service)
def line_on(l, c):
    return 0 if scen[c]["out_line"] == l else 1

def gen_on(g, c):
    return 0 if scen[c]["out_gen"] == g else 1

print(f"INCLUDE_CT={INCLUDE_CT} | INCLUDE_CG={INCLUDE_CG}")
print(f"Eligible: |C_T|={len(CT_lines)} non-radial lines, |C_G|={len(CG_gens)} generators")
print(f"|C| = {len(set_C)} operating states (1 base + {len(set_C) - 1} contingencies)")

INCLUDE_CT=True | INCLUDE_CG=True
Eligible: |C_T|=177 non-radial lines, |C_G|=19 generators
|C| = 197 operating states (1 base + 196 contingencies)


### 3.1 Scenario probabilities $\rho_c$

The base case carries most of the weight; the contingencies share the remainder equally.
The weights must sum to 1. (With only the base case, $\rho_0 = 1$.) These are parameters —
swap in failure-rate-based values later without touching the model structure.

In [23]:
RHO_BASE = 0.99   # weight of the base case when contingencies are present

n_cont = len(set_C) - 1
if n_cont == 0:
    par_rho = {0: 1.0}
else:
    par_rho = {0: RHO_BASE}
    for c in set_C[1:]:
        par_rho[c] = (1.0 - RHO_BASE) / n_cont

assert abs(sum(par_rho.values()) - 1.0) < 1e-9, "rho must sum to 1."
print(f"rho[0] (base) = {par_rho[0]}")
print(f"sum of rho    = {sum(par_rho.values()):.6f}")

rho[0] (base) = 0.99
sum of rho    = 1.000000


## 4. Model and decision variables

All variables carry a scenario index `c` (2-D). Following approach **B**, we give them wide
bounds here and write the physical limits (3.3)–(3.4) as explicit, named constraints, so each
equation of the formulation maps 1:1 to a code block.

- `var_p[g, c]`     — $P_{g,c}$, generator output (≥ 0; upper bound set by (3.3))
- `var_f[l, c]`     — $f_{ij,c}$, branch flow (free; bounded by (3.4))
- `var_theta[i, c]` — $\theta_{i,c}$, bus angle (free; fixed at slack by (3.6))

In [24]:
model = gp.Model("SC-DCOPF_118bus")
model.setParam("OutputFlag", 1)   # set to 0 to silence the solver log

var_p = {
    (g, c): model.addVar(lb=0.0, ub=par_pmax[g], name=f"p[{g},{c}]")
    for g in set_G for c in set_C
}
var_f = {
    (l, c): model.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f"f[{l},{c}]")
    for l in set_L for c in set_C
}
var_theta = {
    (i, c): model.addVar(lb=-GRB.INFINITY, ub=GRB.INFINITY, name=f"theta[{i},{c}]")
    for i in set_N for c in set_C
}

print(f"Created {len(var_p)} p, {len(var_f)} f, {len(var_theta)} theta variables.")

Set parameter OutputFlag to value 1


Created 3743 p, 36642 f, 23246 theta variables.


## 5. Objective — expected generation cost (3.1)

$$\min_{P,f,\theta} \sum_{g,c} \rho_c\, c_g\, P_{g,c}$$

`par_cost` is in \$/MWh and `var_p` is in p.u., so we multiply by `BASE_MVA` to get \$/h.

In [25]:
model.setObjective(
    gp.quicksum(par_rho[c] * par_cost[g] * var_p[(g, c)] * BASE_MVA
                for g in set_G for c in set_C),
    GRB.MINIMIZE,
)

## 6. Constraints

Each block below is one equation of the formulation, written for every scenario `c`.

### (3.2) Nodal power balance

$$\sum_{g \in \mathcal{G}_i} P_{g,c} - P_{d,i} = \sum_{j \in \Omega_i^+} f_{ij,c} - \sum_{j \in \Omega_i^-} f_{ji,c}$$

For each bus: (generation here) minus (load here) equals (flow leaving) minus (flow entering).

In [26]:
for c in set_C:
    for i in set_N:
        gen_i = gp.quicksum(var_p[(g, c)] for g in set_gens_at[i])   # sum over G_i
        out_i = gp.quicksum(var_f[(l, c)] for l in set_out_lines[i]) # Omega_i^+
        in_i  = gp.quicksum(var_f[(l, c)] for l in set_in_lines[i])  # Omega_i^-
        model.addConstr(gen_i - par_pd[i] == out_i - in_i, name=f"bal[{i},{c}]")

### (3.3) Generator limits with availability $N1_{g,c}$

$$P_g^{\min} N1_{g,c} \le P_{g,c} \le P_g^{\max} N1_{g,c}$$

When a generator is out (`gen_on == 0`) both bounds collapse to 0, forcing $P_{g,c}=0$.

In [27]:
for c in set_C:
    for g in set_G:
        avail = gen_on(g, c)
        model.addConstr(var_p[(g, c)] >= par_pmin[g] * avail, name=f"pmin[{g},{c}]")
        model.addConstr(var_p[(g, c)] <= par_pmax[g] * avail, name=f"pmax[{g},{c}]")

### (3.4) Thermal transmission limits

$$-F_{ij}^{\max} \le f_{ij,c} \le F_{ij}^{\max}$$

In [28]:
for c in set_C:
    for l in set_L:
        model.addConstr(var_f[(l, c)] >= -par_rate[l], name=f"fmin[{l},{c}]")
        model.addConstr(var_f[(l, c)] <=  par_rate[l], name=f"fmax[{l},{c}]")

### (3.5)/(3.5b) DC branch power flow

$$f_{ij,c} - \tfrac{1}{x_{ij}}(\theta_{i,c} - \theta_{j,c}) = 0 \;\; (N1_{ij,c}=1), \qquad f_{ij,c} = 0 \;\; (N1_{ij,c}=0)$$

If the line is in service we impose the DC flow law; if it is out, we force $f=0$ **without**
coupling the angles (avoids the 'ghost line' that would tie $\theta_i=\theta_j$).

In [29]:
for c in set_C:
    for l in set_L:
        i, j = par_from[l], par_to[l]
        if line_on(l, c) == 1:
            model.addConstr(
                var_f[(l, c)] == (1.0 / par_x[l]) * (var_theta[(i, c)] - var_theta[(j, c)]),
                name=f"dcflow[{l},{c}]",
            )
        else:
            model.addConstr(var_f[(l, c)] == 0.0, name=f"dcflow_off[{l},{c}]")

### (3.6) Reference bus

$$\theta_{\mathrm{ref},c} = 0$$

In [30]:
for c in set_C:
    model.addConstr(var_theta[(SLACK_BUS, c)] == 0.0, name=f"ref[{c}]")

print(f"Model built: {model.NumVars} variables, {model.NumConstrs} constraints.")

Model built: 0 variables, 0 constraints.


## 7. Solve

In [31]:
model.optimize()

Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-1360P, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 16 logical processors, using up to 16 threads

Academic license 2855642 - for non-commercial use only - registered to ra___@poli.ufrj.br
Optimize a model with 140855 rows, 63631 columns and 267566 nonzeros (Min)
Model fingerprint: 0xb0ff0307
Model has 3743 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 2e+02]
  Objective range  [6e-02, 1e+04]
  Bounds range     [1e-01, 1e+01]
  RHS range        [2e-02, 7e+01]

Presolve removed 123564 rows and 42815 columns
Presolve time: 0.16s
Presolved: 17291 rows, 20816 columns, 80286 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 0.00s

Barrier statistics:
 AA' NZ     : 9.675e+04
 Factor NZ  : 2.189e+05 (roughly 17 MB of memory)
 Factor Ops : 3.298e+

## 8. Results

With $\mathcal{C}=\{0\}$ this is a plain DCOPF, so total generation must equal total load and
the cost is the base-case dispatch cost — a sanity check before switching on contingencies.

In [32]:
if model.status == GRB.OPTIMAL:
    total_load = sum(par_pd.values()) * BASE_MVA
    total_gen  = sum(var_p[(g, 0)].X for g in set_G) * BASE_MVA
    print("=" * 55)
    print("  SC-DCOPF SOLUTION — IEEE 118-Bus")
    print("=" * 55)
    print(f"  Expected cost   : ${model.ObjVal:,.2f} /h")
    print(f"  Total load      : {total_load:.2f} MW")
    print(f"  Total generation: {total_gen:.2f} MW (base case)")
    print(f"  Scenarios       : {len(set_C)}")
elif model.status == GRB.INFEASIBLE:
    print("INFEASIBLE. Computing IIS to locate the conflict...")
    model.computeIIS()
    model.write("scdcopf_infeasible.ilp")
    print("IIS written to scdcopf_infeasible.ilp")
else:
    print(f"Solver status: {model.status}")

INFEASIBLE. Computing IIS to locate the conflict...
Academic license 2855642 - for non-commercial use only - registered to ra___@poli.ufrj.br
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-1360P, instruction set [SSE2|AVX|AVX2]
Thread count: 12 physical cores, 16 logical processors, using up to 16 threads

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0      handle free variables                          0s

IIS computed: 272 constraints and 13 bounds
IIS runtime: 1.73 seconds (1.63 work units)
IIS written to scdcopf_infeasible.ilp


In [33]:
# Base-case dispatch of the active generators
if model.status == GRB.OPTIMAL:
    rows = []
    for g in set_G:
        mw = var_p[(g, 0)].X * BASE_MVA
        if mw > 0.01:
            rows.append({"gen_id": g, "bus": par_g_bus[g],
                         "p_mw": round(mw, 2),
                         "pmax_mw": round(par_pmax[g] * BASE_MVA, 2),
                         "c1": round(par_cost[g], 3)})
    df_dispatch = pd.DataFrame(rows).sort_values("c1").reset_index(drop=True)
    print(f"Active generators: {len(df_dispatch)}")
    df_dispatch